# Retail Analytics ETL Pipeline — Simple Local Python Version

This notebook runs in ordinary Jupyter Notebook on your laptop. It does **not** require PySpark, Hadoop, Java, Azure, or `winutils.exe`.

Keep `sales_data.csv` and `footfall_data.csv` in the same Jupyter folder as these notebooks (your screenshot shows both files in the Jupyter root folder).

Run the notebooks in this order: **Bronze → Silver → Gold**.

## Silver Layer
Clean the source data, remove invalid/duplicate rows, aggregate sales and footfall by store/date/hour, and calculate conversion rate.


In [ ]:
from pathlib import Path
import csv
from datetime import datetime
from collections import defaultdict

BASE_DIR = Path.cwd()
output_root = BASE_DIR / "retail_analytics_output"
bronze_dir = output_root / "bronze"
sales_file = bronze_dir / "sales_data.csv"
footfall_file = bronze_dir / "footfall_data.csv"

if not sales_file.exists() or not footfall_file.exists():
    raise FileNotFoundError("Bronze files not found. Run bronze_notebook.ipynb first.")

def read_csv(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

sales_rows = read_csv(sales_file)
footfall_rows = read_csv(footfall_file)

print("Bronze sales rows:", len(sales_rows))
print("Bronze footfall rows:", len(footfall_rows))


In [ ]:
# Clean sales transactions and aggregate by Store, Date and Hour
sales_hourly = defaultdict(lambda: {"Total_Transactions": 0, "Total_Revenue": 0.0})
seen_receipts = set()

for row in sales_rows:
    try:
        receipt_id = row["Receipt_ID"].strip()
        store_id = row["Store_ID"].strip()
        timestamp = datetime.strptime(row["Timestamp"].strip(), "%Y-%m-%d %H:%M:%S")
        amount = float(row["Total_Amount"])

        if not receipt_id or not store_id or amount < 0 or receipt_id in seen_receipts:
            continue

        seen_receipts.add(receipt_id)
        key = (store_id, timestamp.strftime("%Y-%m-%d"), timestamp.hour)
        sales_hourly[key]["Total_Transactions"] += 1
        sales_hourly[key]["Total_Revenue"] += amount
    except (KeyError, ValueError, TypeError):
        continue

print("Valid unique sales receipts:", len(seen_receipts))
print("Hourly sales groups:", len(sales_hourly))


In [ ]:
# Clean and aggregate footfall by Store, Date and Hour
footfall_hourly = defaultdict(int)
seen_footfall = set()

for row in footfall_rows:
    try:
        store_id = row["Store_ID"].strip()
        timestamp = datetime.strptime(row["Timestamp"].strip(), "%Y-%m-%d %H:%M:%S")
        visitors = int(float(row["Footfall_Count"]))

        key = (store_id, timestamp.strftime("%Y-%m-%d"), timestamp.hour)
        if not store_id or visitors < 0 or key in seen_footfall:
            continue

        seen_footfall.add(key)
        footfall_hourly[key] += visitors
    except (KeyError, ValueError, TypeError):
        continue

print("Valid footfall groups:", len(footfall_hourly))


In [ ]:
# Join sales and footfall; keep all footfall periods
all_keys = sorted(set(footfall_hourly) | set(sales_hourly))
silver_rows = []

for store_id, date, hour in all_keys:
    sales = sales_hourly.get(
        (store_id, date, hour),
        {"Total_Transactions": 0, "Total_Revenue": 0.0}
    )
    visitors = footfall_hourly.get((store_id, date, hour), 0)
    transactions = sales["Total_Transactions"]
    revenue = sales["Total_Revenue"]
    conversion = (transactions / visitors * 100) if visitors > 0 else 0.0

    silver_rows.append({
        "Store_ID": store_id,
        "Date": date,
        "Hour": hour,
        "Total_Transactions": transactions,
        "Total_Revenue": round(revenue, 2),
        "Footfall_Count": visitors,
        "Conversion_Rate_Pct": round(conversion, 2)
    })

print("Silver output rows:", len(silver_rows))
print("Sample rows:")
for row in silver_rows[:5]:
    print(row)


In [ ]:
# Save Silver output as CSV
silver_dir = output_root / "silver"
silver_dir.mkdir(parents=True, exist_ok=True)
silver_file = silver_dir / "hourly_store_performance.csv"

fieldnames = [
    "Store_ID", "Date", "Hour", "Total_Transactions",
    "Total_Revenue", "Footfall_Count", "Conversion_Rate_Pct"
]

with open(silver_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(silver_rows)

print("Silver layer saved:", silver_file)
